<a href="https://colab.research.google.com/github/sruthi-kurra/llm-distillation/blob/main/notebooks/03_distillation_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Setup: load teacher summaries and tokenizer

In [ ]:
!pip install transformers datasets evaluate rouge_score sentencepiece -q

import torch
import json
from transformers import AutoTokenizer

print(f"GPU available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "GPU not available. Switch runtime to T4."
device = torch.device("cuda")

# Load tokenizer (same as teacher, since student uses same vocab)
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

# Upload teacher_summaries.json
from google.colab import files
print("Upload teacher_summaries.json:")
uploaded = files.upload()

with open('teacher_summaries.json', 'r') as f:
    teacher_data = json.load(f)

print(f"\nLoaded {len(teacher_data)} teacher-generated training examples")

# Verify structure
required_keys = ['article', 'reference_summary', 'teacher_summary']
for key in required_keys:
    assert key in teacher_data[0], f"Missing key: {key}"
print("JSON structure verified.")

print(f"Example summary: {teacher_data[0]['teacher_summary'][:100]}...")

from transformers import AutoModelForSeq2SeqLM

print("Loading teacher model for distillation...")
teacher = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
teacher = teacher.to(device)
teacher.eval()
print("Teacher loaded!")

## Cell 2 — Build Mini-BART student architecture

In [ ]:
from transformers import BartConfig, BartForConditionalGeneration
from transformers import AutoConfig

# Load teacher config to inherit vocab size, dimensions
teacher_config = AutoConfig.from_pretrained("facebook/bart-large-cnn")

# Build a smaller student config — fewer layers, same hidden size family
student_config = BartConfig(
    vocab_size=teacher_config.vocab_size,
    max_position_embeddings=teacher_config.max_position_embeddings,
    encoder_layers=3,              # teacher has 12
    decoder_layers=3,              # teacher has 12
    encoder_attention_heads=8,     # teacher has 16
    decoder_attention_heads=8,     # teacher has 16
    encoder_ffn_dim=2048,          # teacher has 4096
    decoder_ffn_dim=2048,          # teacher has 4096
    d_model=512,                   # teacher has 1024
    dropout=0.1,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    decoder_start_token_id=teacher_config.decoder_start_token_id,
    forced_eos_token_id=teacher_config.forced_eos_token_id,
)

student = BartForConditionalGeneration(student_config)
student = student.to(device)

student_params = sum(p.numel() for p in student.parameters())
teacher_params = 406_290_432

print(f"Student model created!")
print(f"Student parameters: {student_params:,} ({student_params/1e6:.1f}M)")
print(f"Teacher parameters: {teacher_params:,} ({teacher_params/1e6:.1f}M)")
print(f"Compression ratio: {teacher_params/student_params:.2f}x smaller")

## Cell 3 — Distillation training loop (CrossEntropy + KL divergence)

In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
import time

# Hyperparameters — reduced for T4 safety
BATCH_SIZE = 2
EPOCHS = 3
LR = 5e-5
ALPHA = 0.5
TEMPERATURE = 2.0
MAX_INPUT_LEN = 384
MAX_TARGET_LEN = 96

optimizer = AdamW(student.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda')

# Teacher is eval-only, freeze gradients
teacher.requires_grad_(False)

def prepare_batch(examples):
    articles = [ex['article'] for ex in examples]
    targets  = [ex['teacher_summary'] for ex in examples]

    inputs = tokenizer(
        articles,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(device)

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(device)

    return inputs, labels

def distillation_loss(student_logits, teacher_logits, labels, alpha=ALPHA, T=TEMPERATURE):
    ce_loss = F.cross_entropy(
        student_logits.view(-1, student_logits.size(-1)),
        labels.view(-1),
        ignore_index=tokenizer.pad_token_id
    )

    student_log_probs = F.log_softmax(student_logits / T, dim=-1)
    teacher_probs      = F.softmax(teacher_logits / T, dim=-1)
    kl_loss = F.kl_div(
        student_log_probs, teacher_probs, reduction='batchmean'
    ) * (T ** 2)

    total_loss = (1 - alpha) * ce_loss + alpha * kl_loss
    return total_loss, ce_loss.item(), kl_loss.item()

# ---- SANITY CHECK: run 1 batch first to confirm no OOM ----
print("Running sanity check on 1 batch...")
test_batch = teacher_data[:BATCH_SIZE]
inputs, labels = prepare_batch(test_batch)

with torch.amp.autocast('cuda'):
    student_outputs = student(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        labels=labels['input_ids']
    )
    student_logits = student_outputs.logits

    with torch.no_grad():
        teacher_outputs = teacher(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            decoder_input_ids=labels['input_ids']
        )
        teacher_logits = teacher_outputs.logits

print(f"Student logits shape: {student_logits.shape}")
print(f"Teacher logits shape: {teacher_logits.shape}")

if student_logits.shape == teacher_logits.shape:
    print("Shapes match! Safe to proceed.")
else:
    print("WARNING: Shape mismatch — need to fix before training!")

print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Reduce dataset for manageable training time
teacher_data = teacher_data[:500]
print(f"Using {len(teacher_data)} samples for training")

# ---- FULL TRAINING LOOP ----
print("\nStarting full distillation training...")
train_losses = []
student.train()

num_batches = len(teacher_data) // BATCH_SIZE
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for batch_idx in range(num_batches):
        batch = teacher_data[batch_idx*BATCH_SIZE : (batch_idx+1)*BATCH_SIZE]
        inputs, labels = prepare_batch(batch)

        optimizer.zero_grad(set_to_none=True)

        try:
            with torch.amp.autocast('cuda'):
                student_outputs = student(
                    input_ids=inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    labels=labels['input_ids']
                )
                student_logits = student_outputs.logits

                with torch.no_grad():
                    teacher_outputs = teacher(
                        input_ids=inputs['input_ids'],
                        attention_mask=inputs['attention_mask'],
                        decoder_input_ids=labels['input_ids']
                    )
                    teacher_logits = teacher_outputs.logits

                loss, ce, kl = distillation_loss(student_logits, teacher_logits, labels['input_ids'])

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            train_losses.append(loss.item())

            if (batch_idx + 1) % 50 == 0:
                elapsed = time.time() - start_time
                print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{num_batches} | "
                      f"Loss: {loss.item():.4f} (CE: {ce:.4f}, KL: {kl:.4f}) | "
                      f"Elapsed: {elapsed:.0f}s")

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM at batch {batch_idx+1}, skipping...")
                torch.cuda.empty_cache()
                continue
            else:
                raise e

    avg_epoch_loss = epoch_loss / num_batches
    print(f"\n=== Epoch {epoch+1} complete | Avg Loss: {avg_epoch_loss:.4f} ===\n")

    # Save checkpoint after every epoch
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': student.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'avg_loss': avg_epoch_loss
    }, f"student_epoch_{epoch+1}.pt")
    print(f"Checkpoint saved: student_epoch_{epoch+1}.pt")

print("Training complete!")

## Cell 4 — Save student checkpoint and training curve

In [ ]:
import json
import os

os.makedirs('results', exist_ok=True)

with open('results/distillation_training_log.json', 'w') as f:
    json.dump({
        'losses': train_losses,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'alpha': ALPHA,
        'temperature': TEMPERATURE,
        'samples_used': len(teacher_data),
        'student_params': student_params,
        'teacher_params': teacher_params,
        'compression_ratio': teacher_params/student_params
    }, f, indent=2)

print("Training log saved!")

from google.colab import files
files.download('student_epoch_3.pt')
files.download('results/distillation_training_log.json')

In [12]:
# Save weights only, in FP16 for smaller size
student_fp16 = student.half()
torch.save(student_fp16.state_dict(), "student_weights_fp16.pt")

import os
size_mb = os.path.getsize("student_weights_fp16.pt") / (1024*1024)
print(f"FP16 weights file size: {size_mb:.1f} MB")

from google.colab import files
files.download("student_weights_fp16.pt")

FP16 weights file size: 93.3 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>